# Diabetic Retinopathy KFold Reproducibility

This notebook reruns and verifies the reviewer-facing diabetic retinopathy experiments:

- patient-level outer train/test split;
- patient-level 5-fold validation on the outer-train set;
- FKG-UM image, FKG-UM table, FKG-MM proposed fusion, and fusion strategy variants;
- full metric set: Sensitivity, Specificity, F1, AUC-ROC, and AUC-PR.

The FKG/FKGS step uses `ROOT_DATA/train_test_selection/train.csv` and the exact `train_kfold/fold_*/train.csv` / `val.csv` manifests created for the deep baselines. That keeps patient IDs grouped and makes the validation folds auditable.

Reviewer formula fixes included in this notebook:

- Let `C = F_img.T @ F_tab / (n - 1)` on the training fold and `C = U S V.T`. The learned projections are `W_img = U[:, :r]` and `W_tab = V[:, :r]`, so both projected modalities have the compatible shape `n x r`.
- Hadamard fusion is `Z_img * Z_tab`, where `Z_img = F_img @ W_img` and `Z_tab = F_tab @ W_tab`. The orthogonal cross-modal objective is `max trace(W_img.T @ C @ W_tab)`.
- The method formerly described ambiguously as Tensor/Kronecker is implemented and named tensor-style cross-SVD fusion: `[Z_img, Z_tab, Z_img * Z_tab]`. It does not claim to be a raw Kronecker product.
- Filter fusion exposes `--filter-corr` and rejects correlated candidates against all already-selected vectors, including both intra-modal and inter-modal pairs.
- Wrapper fusion evaluates the initialized minimum feature set as its baseline, adds a feature only on strict score improvement, and checks `max_img` / `max_tab` before evaluating candidates.

## Environment Setup

The checked-in native FKG extension targets 64-bit Python 3.13 on Windows. Use separate environments because the classical FKG pipeline and the deep baselines have different dependency sets:

```powershell
py -3.13 -m venv .venv
.\.venv\Scripts\python.exe -m pip install -r Source_code\main\diabetic_retinopathy\requirements_fkg_reproducibility.txt
py -3.13 -m venv .venv_deep_baselines
.\.venv_deep_baselines\Scripts\python.exe -m pip install -r Source_code\main\diabetic_retinopathy\requirements_deep_baselines.txt
```

`Run_Diabetic_Retinopathy_KFold_All.bat` and this notebook automatically use `.venv` for split/FKG/reporting and `.venv_deep_baselines` for CNN/MLP training. Set `FKG_PYTHON_EXE` or `DEEP_PYTHON_EXE` to override either interpreter.

In [1]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd

NOTEBOOK_PATH = Path.cwd()
REPO_ROOT = NOTEBOOK_PATH if (NOTEBOOK_PATH / 'Source_code').exists() else NOTEBOOK_PATH.parent
os.chdir(REPO_ROOT)
print('Repository:', REPO_ROOT)

common_python = os.environ.get('PYTHON_EXE')
fkg_venv_python = REPO_ROOT / '.venv' / 'Scripts' / 'python.exe'
deep_venv_python = REPO_ROOT / '.venv_deep_baselines' / 'Scripts' / 'python.exe'
FKG_PYTHON = os.environ.get(
    'FKG_PYTHON_EXE',
    common_python or str(fkg_venv_python if fkg_venv_python.exists() else Path(sys.executable)),
)
DEEP_PYTHON = os.environ.get(
    'DEEP_PYTHON_EXE',
    common_python or str(deep_venv_python if deep_venv_python.exists() else Path(sys.executable)),
)

def show_environment(label: str, executable: str, packages: list[str]) -> None:
    code = (
        "import importlib.metadata as m, platform; "
        "print(platform.python_version()); "
        f"print({{name: m.version(name) for name in {packages!r}}})"
    )
    output = subprocess.check_output([executable, '-c', code], text=True).strip()
    print(f'{label}: {executable}\n{output}')

show_environment('FKG Python', FKG_PYTHON, ['numpy', 'pandas', 'scikit-learn'])
show_environment('Deep Python', DEEP_PYTHON, ['numpy', 'pandas', 'torch', 'torchvision'])

Repository: D:\Study\InternAIRC\source_code_Tan\source_code_Tan


FKG Python: D:\Study\InternAIRC\source_code_Tan\source_code_Tan\.venv\Scripts\python.exe
3.13.7
{'numpy': '2.4.1', 'pandas': '2.3.3', 'scikit-learn': '1.8.0'}


Deep Python: D:\Study\InternAIRC\source_code_Tan\source_code_Tan\.venv_deep_baselines\Scripts\python.exe
3.13.7
{'numpy': '2.5.2', 'pandas': '3.0.5', 'torch': '2.14.0+cu130', 'torchvision': '0.29.0+cu130'}


In [2]:
RUN_ID = os.environ.get('RUN_ID', '20260921')
RUN_TAG = os.environ.get('RUN_TAG', f'kfold_rerun_{RUN_ID}_formula_fix')
DEVICE = os.environ.get('DEVICE', 'auto')
RESNET_ARCH = os.environ.get('RESNET_ARCH', 'resnet50')
EPOCHS = int(os.environ.get('EPOCHS', '10'))
BATCH_SIZE = int(os.environ.get('BATCH_SIZE', '16'))
FIS_BACKEND = os.environ.get('FIS_BACKEND', 'cpu')
FKG_BACKEND = os.environ.get('FKG_BACKEND', 'auto')
FKGS_RAN = os.environ.get('FKGS_RAN', '15 20').split()
FKGS_EPSILON = os.environ.get('FKGS_EPSILON', '0.2 0.3').split()
FKGS_TURNS = os.environ.get('FKGS_TURNS', '1')
FKGS_WORKERS = os.environ.get('FKGS_WORKERS', '1')
FKG_MODALITIES = os.environ.get(
    'FKG_MODALITIES',
    'table image fusion fusion_filter fusion_hadamard fusion_tensor fusion_wrapper',
).split()

SPLIT_ROOT = Path('ROOT_DATA/train_test_selection')
DEEP_RESULTS = SPLIT_ROOT / 'deep_baselines' / f'kfold_rerun_{RUN_ID}'
FKGS_OUTPUT = Path('Source_code/data/Dataset_diabetic') / f'KFold_feature_selection_rerun_{RUN_ID}'
FKGS_REPORT = Path('Source_code/data/result') / f'KFold_feature_selection_rerun_{RUN_ID}'
COMPARISON_STEM = Path('result') / f'diabetic_retinopathy_model_comparison_kfold_rerun_{RUN_ID}'

RESTRICT_IMAGE_IDS = SPLIT_ROOT / 'train.csv'
FOLD_MANIFEST_ROOT = SPLIT_ROOT / 'train_kfold'
PROTOCOL_TEXT = (
    'Patient-grouped 5-fold CV with shared deep-baseline train image IDs; '
    'fold_manifest_root=ROOT_DATA/train_test_selection/train_kfold; '
    'formula_fix cross-SVD fusion.'
)

print('RUN_ID:', RUN_ID)
print('RUN_TAG:', RUN_TAG)
print('Modalities:', FKG_MODALITIES)
print('Deep results:', DEEP_RESULTS)
print('FKG/FKGS report:', FKGS_REPORT)
print('Comparison stem:', COMPARISON_STEM)

RUN_ID: 20260921
RUN_TAG: kfold_rerun_20260921_formula_fix
Modalities: ['table', 'image', 'fusion', 'fusion_filter', 'fusion_hadamard', 'fusion_tensor', 'fusion_wrapper']
Deep results: ROOT_DATA\train_test_selection\deep_baselines\kfold_rerun_20260921
FKG/FKGS report: Source_code\data\result\KFold_feature_selection_rerun_20260921
Comparison stem: result\diabetic_retinopathy_model_comparison_kfold_rerun_20260921


In [3]:
def run_command(command: list[str], *, skip_if: Path | None = None, force: bool = False) -> None:
    if skip_if is not None and skip_if.exists() and not force:
        print(f'[SKIP] {skip_if} already exists')
        return
    print('[RUN]', ' '.join(str(part) for part in command))
    subprocess.run([str(part) for part in command], check=True, cwd=REPO_ROOT)

def read_json(path: Path) -> dict:
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)

## 1. Create Patient-Aware Splits

This cell is fast. It writes `ROOT_DATA/train_test_selection/summary.json`, `train.csv`, `test.csv`, and `train_kfold/fold_*` manifests. The split is patient-grouped and greedily stratified by label counts. The assertions below fail if any patient appears on both sides of a split. Install the classical pipeline with `requirements_fkg_reproducibility.txt`; the CNN environment uses `requirements_deep_baselines.txt`.

In [4]:
run_command([
    FKG_PYTHON,
    'Source_code/main/diabetic_retinopathy/create_root_data_image_train_test_split.py',
    '--root-data', 'ROOT_DATA',
    '--image-dir', 'ROOT_DATA/fundus_photos_224',
    '--materialize', 'none',
    '--path-mode', 'relative',
    '--overwrite',
])

[RUN] D:\Study\InternAIRC\source_code_Tan\source_code_Tan\.venv\Scripts\python.exe Source_code/main/diabetic_retinopathy/create_root_data_image_train_test_split.py --root-data ROOT_DATA --image-dir ROOT_DATA/fundus_photos_224 --materialize none --path-mode relative --overwrite


In [5]:
split_summary = read_json(SPLIT_ROOT / 'summary.json')
assert split_summary['patient_overlap_count'] == 0, split_summary['patient_overlap_count']
for fold in split_summary['train_kfold']['folds']:
    assert fold['patient_overlap_count'] == 0, fold
print('Outer split patients:', split_summary['train']['patients'], '/', split_summary['test']['patients'])
print('Outer split images:', split_summary['train']['images'], '/', split_summary['test']['images'])
pd.DataFrame(split_summary['train_kfold']['folds'])[['fold', 'patient_overlap_count']]

Outer split patients: 729 / 168
Outer split images: 1208 / 321


,fold,patient_overlap_count
0,1,0
1,2,0
2,3,0
3,4,0
4,5,0


## 2. Run Deep Baselines

This is the slow CNN/MLP step. It is skipped when `summary.csv` already exists for the chosen `RUN_ID`; delete the output folder or pass `force=True` to rerun.

In [6]:
run_command([
    DEEP_PYTHON,
    'Source_code/main/diabetic_retinopathy/run_deep_multimodal_baselines.py',
    '--split-root', str(SPLIT_ROOT),
    '--tabular-csv', 'Source_code/data/Dataset_diabetic/data_process.csv',
    '--results-dir', str(DEEP_RESULTS),
    '--models', 'all',
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--device', DEVICE,
    '--resnet-arch', RESNET_ARCH,
], skip_if=DEEP_RESULTS / 'summary.csv')

[SKIP] ROOT_DATA\train_test_selection\deep_baselines\kfold_rerun_20260921\summary.csv already exists


## 3. Run FIS + Native FKG + FKGS

This cell uses the same outer-train images and the exact same fold manifests as the deep baseline. It covers FKG-UM image/table, FKG-MM proposed fusion, and the filter/hadamard/tensor/wrapper fusion strategies.

In [7]:
fkg_command = [
    FKG_PYTHON,
    'Source_code/main/diabetic_retinopathy/Preprocess_kfold_feature_selection.py',
    '--modalities', *FKG_MODALITIES,
    '--run-fkgs',
    '--reuse-fkgs',
    '--run-fkg',
    '--fis-engine', 'native',
    '--native-backend', FIS_BACKEND,
    '--fkg-backend', FKG_BACKEND,
    '--restrict-image-ids', str(RESTRICT_IMAGE_IDS),
    '--fold-manifest-root', str(FOLD_MANIFEST_ROOT),
    '--run-tag', RUN_TAG,
    '--ran', *FKGS_RAN,
    '--e', *FKGS_EPSILON,
    '--fkgs-turns', FKGS_TURNS,
    '--fkgs-workers', FKGS_WORKERS,
    '--output-root', str(FKGS_OUTPUT.relative_to('Source_code')),
    '--report-root', str(FKGS_REPORT.relative_to('Source_code')),
]
run_command(fkg_command, skip_if=FKGS_REPORT / 'kfold_modality_mean_std_summary.csv')

[SKIP] Source_code\data\result\KFold_feature_selection_rerun_20260921\kfold_modality_mean_std_summary.csv already exists


## 4. Build Comparison Table

The output CSV contains the full metric columns; the Markdown file is a compact reviewer table.

In [8]:
run_command([
    FKG_PYTHON,
    'Source_code/main/diabetic_retinopathy/collect_kfold_model_comparison.py',
    '--deep-summary', str(DEEP_RESULTS / 'summary.csv'),
    '--deep-config', str(DEEP_RESULTS / 'config.json'),
    '--fkgs-summary', str(FKGS_REPORT / 'kfold_fkgs_mean_std_summary.csv'),
    '--fkgs-tables', str(FKGS_REPORT / 'kfold_fkgs_tables.csv'),
    '--fkg-summary', str(FKGS_REPORT / 'kfold_modality_mean_std_summary.csv'),
    '--output-stem', str(COMPARISON_STEM),
    '--protocol', PROTOCOL_TEXT,
], skip_if=COMPARISON_STEM.with_suffix('.csv'))

[SKIP] result\diabetic_retinopathy_model_comparison_kfold_rerun_20260921.csv already exists


In [9]:
comparison_csv = COMPARISON_STEM.with_suffix('.csv')
comparison_md = COMPARISON_STEM.with_suffix('.md')
comparison = pd.read_csv(comparison_csv)
display_columns = [
    'model', 'data_type', 'source_family',
    'accuracy_pct', 'sensitivity_pct', 'specificity_pct',
    'precision_pct', 'f1_pct', 'auc_roc_pct', 'auc_pr_pct',
]
display(comparison[display_columns])
print('CSV:', comparison_csv.resolve())
print('Markdown:', comparison_md.resolve())

,model,data_type,source_family,accuracy_pct,sensitivity_pct,specificity_pct,precision_pct,f1_pct,auc_roc_pct,auc_pr_pct
0,MLP,Tabular,deep_baseline_KFold,86.014197,79.473684,86.593860,36.682066,49.246991,92.644926,57.201796
1,ResNet-50,Image,deep_baseline_KFold,69.551799,55.789474,70.773989,16.597964,23.743699,73.035782,25.720564
2,Early Fusion (MLP),Multimodal,deep_baseline_KFold,88.578581,66.052632,90.554614,39.205488,48.720184,89.225212,55.759264
3,Late Fusion (Ensemble),Multimodal,deep_baseline_KFold,86.847159,74.052632,87.947661,39.992708,49.127690,90.884657,49.650932
4,FKG-UM (Bảng),Unimodal FKG,FKG_KFold,89.816879,40.421053,94.151379,36.595238,37.261170,80.567758,32.788752
5,FKG-UM (Ảnh),Unimodal FKG,FKG_KFold,74.258770,33.000000,77.867631,10.796546,15.953456,58.557259,12.428683
6,FKG-MM (đề xuất),Multimodal FKG,FKG_KFold,91.722163,11.421053,98.741563,39.090909,16.618182,80.729487,24.533841
7,FKG-MM (Filter),Multimodal FKG,FKG_KFold,91.722849,13.526316,98.562191,44.519481,19.704571,76.035047,20.621683
8,FKG-MM (Hadamard),Multimodal FKG,FKG_KFold,62.688179,44.473684,64.285777,10.657805,16.760545,56.369414,10.828470
9,FKG-MM (Tensor),Multimodal FKG,FKG_KFold,91.638147,2.105263,99.459860,10.000000,3.478261,55.742178,9.590936


CSV: D:\Study\InternAIRC\source_code_Tan\source_code_Tan\result\diabetic_retinopathy_model_comparison_kfold_rerun_20260921.csv
Markdown: D:\Study\InternAIRC\source_code_Tan\source_code_Tan\result\diabetic_retinopathy_model_comparison_kfold_rerun_20260921.md


## 5. Inspect Both Reviewer Scenarios

Scenario 1 compares FKG-UM table, FKG-UM image, and the proposed FKG-MM model. Scenario 2 compares the proposed fusion/feature-selection method with Filter, Hadamard, Tensor-style cross-SVD, and Wrapper fusion. Every `ran`/`epsilon` configuration includes the full requested metric set.

In [10]:
scenario_tables_csv = FKGS_REPORT / 'kfold_fkgs_tables.csv'
scenario_tables = pd.read_csv(scenario_tables_csv)
scenario_metrics = [
    'accuracy_pct', 'sensitivity_pct', 'specificity_pct',
    'f1_pct', 'auc_roc_pct', 'auc_pr_pct',
]
scenario_std_metrics = [f'{column[:-4]}_std_pct' for column in scenario_metrics]
missing = [
    column for column in scenario_metrics + scenario_std_metrics
    if column not in scenario_tables.columns
]
assert not missing, f'Missing scenario metrics: {missing}'

scenario_1_keys = ['table', 'image', 'fusion']
scenario_2_keys = [
    'fusion', 'fusion_filter', 'fusion_hadamard',
    'fusion_tensor', 'fusion_wrapper',
]
scenario_columns = [
    'table', 'ran', 'epsilon', 'model', 'modality',
    *scenario_metrics, 'train_time_s', 'test_time_s', 'end_to_end_time_s',
]

print('Scenario 1: FKG-MM versus FKG-UM image/table')
display(scenario_tables[scenario_tables['modality_key'].isin(scenario_1_keys)][scenario_columns])
print('Scenario 2: fusion strategy comparison')
display(scenario_tables[scenario_tables['modality_key'].isin(scenario_2_keys)][scenario_columns])
print('Scenario CSV:', scenario_tables_csv.resolve())

Scenario 1: FKG-MM versus FKG-UM image/table


,table,ran,epsilon,model,modality,accuracy_pct,sensitivity_pct,specificity_pct,f1_pct,auc_roc_pct,auc_pr_pct,train_time_s,test_time_s,end_to_end_time_s
0,Bang 3.2,15,0.2,FKG-UM,Du lieu dang bang full,91.060,21.421053,97.117488,24.042042,72.210715,24.274380,3.547577,6.813968,10.361544
1,Bang 3.2,15,0.2,FKG-UM,Du lieu anh,70.620,36.789474,73.533119,15.143205,55.559464,11.633090,2.517703,4.702932,7.220634
2,Bang 3.2,15,0.2,FKG-MM,Du lieu anh+bang,91.642,6.210526,99.101519,10.156510,65.383353,14.431690,5.345535,11.236923,16.582459
7,Bang 3.3,15,0.3,FKG-UM,Du lieu dang bang full,90.482,23.789474,96.311125,23.799847,73.595809,24.471872,3.555295,6.847593,10.402888
8,Bang 3.3,15,0.3,FKG-UM,Du lieu anh,65.150,39.052632,67.414199,14.899027,56.663641,12.758092,2.521574,4.633549,7.155123
9,Bang 3.3,15,0.3,FKG-MM,Du lieu anh+bang,91.558,2.052632,99.370982,3.571429,60.670701,11.061366,5.359897,11.086789,16.446686
14,Bang 3.4,20,0.2,FKG-UM,Du lieu dang bang full,90.066,27.000000,95.587955,26.924067,75.128819,23.520349,4.074729,7.930141,12.004869
15,Bang 3.4,20,0.2,FKG-UM,Du lieu anh,66.226,37.947368,68.680741,15.122697,53.900726,12.909472,2.583478,4.815510,7.398989
16,Bang 3.4,20,0.2,FKG-MM,Du lieu anh+bang,91.640,6.210526,99.101115,9.790210,69.407204,15.296815,6.264137,13.829082,20.093219
21,Bang 3.5,20,0.3,FKG-UM,Du lieu dang bang full,90.728,24.684211,96.488881,26.645022,71.363711,22.919513,4.069612,7.887507,11.957119


Scenario 2: fusion strategy comparison


,table,ran,epsilon,model,modality,accuracy_pct,sensitivity_pct,specificity_pct,f1_pct,auc_roc_pct,auc_pr_pct,train_time_s,test_time_s,end_to_end_time_s
2,Bang 3.2,15,0.2,FKG-MM,Du lieu anh+bang,91.642,6.210526,99.101519,10.156510,65.383353,14.431690,5.345535,11.236923,16.582459
3,Bang 3.2,15,0.2,FKG-MM,Fusion Filter,91.556,3.105263,99.281296,5.484848,61.650841,12.071122,5.377128,12.447585,17.824713
4,Bang 3.2,15,0.2,FKG-MM,Fusion Hadamard,76.074,30.947368,80.017979,16.846801,57.504156,11.781797,2.466440,4.378278,6.844717
5,Bang 3.2,15,0.2,FKG-MM,Fusion Tensor,91.640,0.000000,99.639232,0.000000,50.224825,8.090478,57.548020,43.528361,101.076381
6,Bang 3.2,15,0.2,FKG-MM,Fusion Wrapper,80.140,38.000000,83.817279,25.191106,65.865323,20.891440,28.031618,3.754447,31.786065
9,Bang 3.3,15,0.3,FKG-MM,Du lieu anh+bang,91.558,2.052632,99.370982,3.571429,60.670701,11.061366,5.359897,11.086789,16.446686
10,Bang 3.3,15,0.3,FKG-MM,Fusion Filter,91.640,5.157895,99.190398,8.550725,58.326681,11.592931,5.415157,12.238639,17.653796
11,Bang 3.3,15,0.3,FKG-MM,Fusion Hadamard,74.094,32.894737,77.674035,16.435660,57.830017,11.202633,2.466290,4.605221,7.071511
12,Bang 3.3,15,0.3,FKG-MM,Fusion Tensor,92.054,1.000000,100.000000,1.904762,50.902551,9.007360,57.898926,49.241512,107.140438
13,Bang 3.3,15,0.3,FKG-MM,Fusion Wrapper,78.398,28.789474,82.733348,18.833230,59.334284,18.303633,28.033293,3.706586,31.739879


Scenario CSV: D:\Study\InternAIRC\source_code_Tan\source_code_Tan\Source_code\data\result\KFold_feature_selection_rerun_20260921\kfold_fkgs_tables.csv


## 6. Audit Fold Artifacts

The cells below are non-training checks for reviewers. They verify that the FKG summary rows came from patient-level folds with zero overlap and that the full metric columns are present.

In [11]:
run_summary = pd.read_csv(FKGS_REPORT / 'kfold_run_summary.csv')
fkgs_run_summary = pd.read_csv(FKGS_REPORT / 'kfold_fkgs_run_summary.csv')
expected_modalities = {
    'table', 'image', 'fusion', 'fusion_filter',
    'fusion_hadamard', 'fusion_tensor', 'fusion_wrapper',
}
assert len(run_summary) == 35, f'Expected 35 native FKG fold rows, got {len(run_summary)}'
assert len(fkgs_run_summary) == 140, f'Expected 140 FKGS fold/config rows, got {len(fkgs_run_summary)}'
assert set(run_summary['modality']) == expected_modalities
assert set(fkgs_run_summary['modality']) == expected_modalities
assert (run_summary.groupby('modality').size() == 5).all()
assert (fkgs_run_summary.groupby(['modality', 'ran', 'epsilon']).size() == 5).all()
assert (run_summary['patient_overlap_count'].fillna(0).astype(int) == 0).all()
assert (fkgs_run_summary['patient_overlap_count'].fillna(0).astype(int) == 0).all()
assert set(run_summary['splitter']) == {'external_manifest_train_val'}
required_columns = ['fkg_sensitivity', 'fkg_specificity', 'fkg_f1', 'fkg_auc_roc', 'fkg_auc_pr']
missing = [column for column in required_columns if column not in run_summary.columns]
assert not missing, missing
print('Native FKG fold rows:', len(run_summary))
print('FKGS fold/config rows:', len(fkgs_run_summary))
print('Modalities:', sorted(run_summary['modality'].dropna().unique()))
run_summary[['modality', 'fold', 'splitter', 'patient_overlap_count']].head(10)

Native FKG fold rows: 35
FKGS fold/config rows: 140
Modalities: ['fusion', 'fusion_filter', 'fusion_hadamard', 'fusion_tensor', 'fusion_wrapper', 'image', 'table']


,modality,fold,splitter,patient_overlap_count
0,table,1,external_manifest_train_val,0
1,table,2,external_manifest_train_val,0
2,table,3,external_manifest_train_val,0
3,table,4,external_manifest_train_val,0
4,table,5,external_manifest_train_val,0
5,image,1,external_manifest_train_val,0
6,image,2,external_manifest_train_val,0
7,image,3,external_manifest_train_val,0
8,image,4,external_manifest_train_val,0
9,image,5,external_manifest_train_val,0
